# 15 - Analysis of the bus network alone

The national graph built in `02_graph_construction` mixes together every transport mode in the Israeli GTFS feed: bus, heavy rail, light rail, cable car, trolleybus, and demand-responsive service. Bus is the dominant mode by a wide margin - about 97% of all scheduled trips - so it's tempting to assume that the all-modes network *is* the bus network. This notebook builds the bus-only graph from the raw feed and tests that assumption instead of taking it for granted.

Concretely it: streams `stop_times.txt` while keeping only trips whose route satisfies `route_type = 3` (regular bus), builds the bus trip-adjacency graph, measures its global structure, computes degree / weighted degree / PageRank / sampled betweenness, extracts its articulation points and bridges, ranks the top bus stations, runs a targeted-versus-random node removal experiment, and finally compares the bus graph against the all-modes graph node by node - and in particular, which stations stop or start being cut vertices the moment you remove the trains and the other secondary modes. Demand-responsive bus (`route_type = 715`) is counted and reported separately rather than merged into the bus network, because it's a different service model (14 routes / 458 trips) and merging it would quietly change what "bus" means.

**Research question:** is the bus network structurally interchangeable with the national multi-modal network, or does the small non-bus layer (mostly trains) meaningfully change where the single points of failure sit?

## Input

* `israel-public-transportation/routes.txt`, `trips.txt` - to map every `trip_id` to its GTFS mode (the mode lives in `routes.txt`; we join `trips -> routes` on `route_id`).
* `israel-public-transportation/stop_times.txt` - 816 MB / 15.7M rows, not tracked in git, downloaded on demand from Google Drive by a cell below and read line by line, never loaded as a table.
* `outputs/nb/02_graph_construction/tables/nodes.csv` - station attributes (`stop_id, stop_name, lat, lon, region, metro, stop_use_count`).
* `outputs/nb/02_graph_construction/tables/edges.csv` - the directed all-modes segments (`from_stop, to_stop, trip_frequency`), used as the comparison baseline.
* *Optional cross-checks, skipped with a printed message if absent:* `outputs/nb/03_descriptive_analysis/tables/articulation_points.csv` and `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv`.

## Notebooks that must run first

1. `01_data_preparation` (produces the cleaned station table that 02 uses),
2. `02_graph_construction` (required - this notebook reads its `nodes.csv` and `edges.csv`).
3. `03_descriptive_analysis` and `04_centrality_analysis` are optional; they're used only to cross-check numbers this notebook recomputes on its own.

## Output (all under `outputs/nb/15_bus_network/`)

| Path | Contents |
|---|---|
| `bus_graph_undirected.pkl` | pickled `networkx.Graph` - the bus-only network, undirected, edge attribute `weight` = bus trips per segment |
| `bus_summary.json` | every key number produced here, plus the constants that were used |
| `tables/bus_station_metrics.csv` | `stop_id, stop_name, lat, lon, region, degree, weighted_degree, approx_betweenness, is_articulation_point` (+ extras) |
| `tables/bus_edges.csv` | the directed bus segments, so no downstream step has to stream 816 MB again |
| `tables/top_bus_stations.csv` | union of the top N stations under four rankings |
| `tables/bus_articulation_points.csv`, `tables/bus_bridges.csv` | the bus-only cut vertices and cut edges |
| `tables/bus_resilience.csv` | targeted-versus-random node removal curves |
| `tables/bus_vs_allmode_summary.csv` | bus structure and all-modes structure side by side |
| `tables/articulation_point_comparison.csv` | every station that is a cut vertex in one network and not the other |
| `tables/demand_responsive_summary.csv` | the `route_type = 715` layer, reported separately |
| `figures/top_bus_stations.png`, `figures/bus_resilience_curves.png`, `figures/bus_vs_allmode.png` | the three figures |

Nothing outside `outputs/nb/15_bus_network/` is written.

## 1. Set up the working environment

The cell below lets the notebook run both on a local copy and on Google Colab. It defines `_ensure(...)`, which pip-installs only the packages that are actually missing (so re-running the notebook is cheap), and `find_repo_root()`, which climbs upward from the current directory looking for the GTFS folder and, if it can't find one, clones the repository into `/content`. It then sets `REPO`, `DATA`, and `OUT` and creates the notebook's output root. Every later cell relies on these three paths, so this is the cell that has to run first.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Libraries, stage folders, and cost knobs

We import the scientific stack, fix the folder layout for this stage (`outputs/nb/15_bus_network/` with `tables/` and `figures/`), and gather every expensive setting in one place, so a reviewer can trade runtime for accuracy without hunting through the notebook.

What each constant costs:

* `K_BETWEENNESS = 300` - the number of pivot sources for `networkx.betweenness_centrality(k=...)`. Exact betweenness on a graph of about 29k nodes would need 29k single-source shortest-path sweeps (hours); 300 pivots take roughly 1-3 minutes per graph. We pay this twice - once for the bus graph and once for the all-modes graph - because comparing our sampled bus values against a table that was sampled differently in another notebook wouldn't be a fair comparison. It's an estimate: the top of the ranking is fairly stable at k=300, the long tail is noisy. Raise the value if you want tighter tail estimates.
* `COMPUTE_ALLMODE_BETWEENNESS = True` - set to `False` to skip the second betweenness run (saves half the cost, and drops the betweenness part of the bus-versus-all-modes comparison).
* `RANDOM_TRIALS = 10` and `REMOVAL_COUNTS` - the node removal experiment. Each (strategy, removal count) pair costs one connected-components pass over the surviving graph, so the total is `(3 targeted + 10 random) x 10 points` ~ 130 passes, well under a minute.
* `PROGRESS_EVERY = 2_000_000` - the progress-print interval for the streaming pass. Cosmetic.
* The streaming pass itself is the real cost of this notebook: 3-6 minutes for one sequential read of 15.7M rows.
* `SEED = 42` fixes the betweenness pivots and the random removal orders, so re-running reproduces the same numbers.

In [ ]:
# --- Libraries, stage folders and cost knobs ------------------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn', 'scipy')

import csv, json, pickle, random, time
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

sns.set_theme(style='whitegrid', font_scale=1.05)
csv.field_size_limit(10_000_000)   # a handful of rows in stop_times.txt are very long

# --- GTFS mode codes (verified against this feed) -------------------------
BUS_ROUTE_TYPE = '3'      # ordinary bus  : 6,796 routes / 412,544 trips
DRT_ROUTE_TYPE = '715'    # demand-responsive bus : 14 routes / 458 trips

# --- Cost knobs (see the markdown above) ----------------------------------
K_BETWEENNESS = 300                  # betweenness pivots; ~1-3 min per graph
COMPUTE_ALLMODE_BETWEENNESS = True   # False -> skip the second betweenness run
SEED = 42                            # pivots + random removal orderings
RANDOM_TRIALS = 10                   # Monte-Carlo repetitions of random failure
REMOVAL_COUNTS = [0, 10, 25, 50, 100, 250, 500, 1000, 2000, 3000]
PROGRESS_EVERY = 2_000_000           # progress print interval of the streaming pass
FIG_DPI = 150                        # figure resolution
TOP_N = 20                           # rows in every top-N table / bar chart

# --- This stage's own output folder ---------------------------------------
STAGE = OUT / '15_bus_network'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print('networkx', nx.__version__, '| pandas', pd.__version__)
print('stage folder:', STAGE)

## 3. Handling Hebrew labels

Station names in the Israeli GTFS feed are in Hebrew, and several figures below print them. Matplotlib doesn't implement the Unicode bidirectional algorithm, so right-to-left text comes out reversed and unreadable. The cell below applies a one-time monkey-patch to `matplotlib.text.Text.set_text` so that any string containing Hebrew characters is converted to display order via `python-bidi` before it's drawn, and it picks a font that actually contains Hebrew glyphs (Arial on Windows, DejaVu Sans everywhere else). The operation is idempotent - re-running won't stack patches on top of each other. Because the patch is global, pass matplotlib raw Hebrew strings from here on; calling `fix_he()` by hand as well would reverse the text twice. All other text in the notebook is in English.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Load the outputs of notebook 02

Two things come from the earlier stages: the station attribute table (`nodes.csv`), which gives each bus station its Hebrew name, coordinates, district, and metropolitan area, and the all-modes edge list (`edges.csv`), which is the baseline the bus network is compared against in section 11.

Stage folders are identified by their two-digit prefix (`OUT.glob('02*')`) rather than an exact slug, so renaming a folder doesn't break the notebook, and the output itself is searched for recursively inside the stage folder (it may sit at the root or under `tables/`). If it's missing, `load_stage_table` raises a `FileNotFoundError` that names the notebook you need to run first - failing quietly here would produce an attribute-less graph and meaningless maps ten cells later. `try_load_stage_table` is the soft variant used for the optional cross-checks against notebooks 03 and 04.

In [ ]:
# --- Locating artifacts written by earlier notebooks ----------------------
def find_stage(prefix, notebook_hint):
    """Return the stage folder whose name starts with `prefix` (e.g. '02')."""
    matches = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir())
    if not matches:
        raise FileNotFoundError(
            f'No stage folder starting with "{prefix}" under {OUT} - '
            f'run notebook {notebook_hint} first.')
    return matches[0]


def find_artifact(stage_dir, filename):
    """Path of `filename` anywhere under a stage folder, or None if absent."""
    direct = stage_dir / filename
    if direct.exists():
        return direct
    matches = sorted(stage_dir.rglob(filename))
    return matches[0] if matches else None


def load_stage_table(prefix, filename, notebook_hint, **kwargs):
    """Read a CSV produced by an earlier stage; fail with an actionable message."""
    stage_dir = find_stage(prefix, notebook_hint)
    path = find_artifact(stage_dir, filename)
    if path is None:
        raise FileNotFoundError(
            f'{filename} not found under {stage_dir} - run notebook {notebook_hint} '
            f'first; it writes {filename}.')
    table = pd.read_csv(path, encoding='utf-8-sig', **kwargs)
    print(f'loaded {filename}: {len(table):,} rows  <-  {path}')
    return table


def try_load_stage_table(prefix, filename, notebook_hint, **kwargs):
    """Optional variant: returns None and prints a note instead of raising."""
    try:
        return load_stage_table(prefix, filename, notebook_hint, **kwargs)
    except FileNotFoundError as exc:
        print('OPTIONAL INPUT MISSING -', exc)
        return None


nodes_all = load_stage_table('02', 'nodes.csv', '02_graph_construction',
                             dtype={'stop_id': str})
edges_all = load_stage_table('02', 'edges.csv', '02_graph_construction',
                             dtype={'from_stop': str, 'to_stop': str})
nodes_all.head()

## 5. External data dependency: `stop_times.txt`

The file `stop_times.txt` weighs 816 MB - well over GitHub's file-size limit - so it isn't in the repository. The cell below downloads it from Google Drive on the first run and skips the download if the file already exists. This is the notebook's only external network dependency. The download takes a few minutes on a first Colab run.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## 6. Which trips are bus trips?

GTFS stores the mode on the route, not the trip, so "bus trip" has to be derived: read `routes.txt`, keep `route_type = 3`, then keep every `trip_id` in `trips.txt` whose `route_id` points at one of those routes. The same is done for `route_type = 715` (demand-responsive bus), which we deliberately keep as a separate layer.

The cell also prints the full mode inventory (routes and trips per `route_type`), so the 97%-of-trips claim in the introduction is verified from the data rather than just asserted, and it reports how many trips carry a `route_id` that doesn't appear in `routes.txt` at all - this number should be zero, and if it isn't, the mode assignment is incomplete and every count below is a lower bound.

In [ ]:
# --- Map every trip_id to its GTFS mode -----------------------------------
routes = pd.read_csv(DATA / 'routes.txt', dtype=str, encoding='utf-8-sig')
trips = pd.read_csv(DATA / 'trips.txt', dtype=str, encoding='utf-8-sig')

MODE_LABELS = {'0': 'tram / light rail', '2': 'rail', '3': 'bus',
               '5': 'cable tram', '8': 'trolleybus',
               '715': 'demand-responsive bus'}

route_type_of = dict(zip(routes['route_id'], routes['route_type']))
trips['route_type'] = trips['route_id'].map(route_type_of)
unmapped_trips = int(trips['route_type'].isna().sum())

inventory = (trips.groupby('route_type').size().rename('trips').to_frame()
             .join(routes.groupby('route_type').size().rename('routes'))
             .reset_index())
inventory['mode_label'] = inventory['route_type'].map(MODE_LABELS).fillna('other')
inventory['trip_share'] = (inventory['trips'] / inventory['trips'].sum()).round(4)
inventory = (inventory[['route_type', 'mode_label', 'routes', 'trips', 'trip_share']]
             .sort_values('trips', ascending=False).reset_index(drop=True))

bus_trip_ids = set(trips.loc[trips['route_type'] == BUS_ROUTE_TYPE, 'trip_id'])
drt_trip_ids = set(trips.loc[trips['route_type'] == DRT_ROUTE_TYPE, 'trip_id'])
if not bus_trip_ids:
    raise RuntimeError('No route_type=3 (bus) trips found - check routes.txt / trips.txt.')

# trip_id -> layer label used by the streaming pass below
TRIP_MODE = {tid: 'bus' for tid in bus_trip_ids}
TRIP_MODE.update({tid: 'drt' for tid in drt_trip_ids})

print(f'bus trips (route_type {BUS_ROUTE_TYPE})        : {len(bus_trip_ids):,} '
      f'({len(bus_trip_ids) / len(trips):.2%} of all trips)')
print(f'demand-responsive trips ({DRT_ROUTE_TYPE})     : {len(drt_trip_ids):,}')
print(f'trips with a route_id absent from routes.txt : {unmapped_trips:,}')
inventory

## 7. The streaming pass over 15.7M rows

This is the expensive step (3-6 minutes) and the only place we touch the 816 MB feed. `csv.reader` returns one row at a time; the only state kept is the segment counters (about 50k records), the per-station stop counts, and the previous row's trip / stop / sequence. Memory is `O(|E|)`, never `O(rows)`.

The edge rule is the same as notebook 02: if the current row belongs to the same trip as the previous row and the two stops differ, increment `W(prev_stop, stop)`. The filter is applied per trip, not per row - either all of a trip's rows are kept or none of them are - so consecutive kept rows of a kept trip are still consecutive stops of the same trip, and notebook 02's contiguity assumption carries over unchanged. Trips of other modes are skipped after a single dictionary lookup done once per trip block, not once per row.

Two fail-loud guards come along at no extra cost: self-loops (a trip listing the same stop twice in a row) are counted and skipped rather than turned into meaningless loop edges, and `stop_sequence` regressions within a trip block are counted. A non-zero regression count means the feed isn't sorted by `(trip_id, stop_sequence)` and the segments produced here would be wrong without anyone noticing; in that case the cell prints a prominent warning.

Note that we don't parse `arrival_time` / `departure_time` at all here. This notebook is purely a topological study; the GTFS "hours >= 24" caveat (`25:30:00` = 01:30 on the next service day) is only relevant to the travel-time notebooks.

In [ ]:
# --- One streaming pass over stop_times.txt, counting segments per mode ----
def stream_mode_edges(path, trip_mode, progress_every=PROGRESS_EVERY):
    """Count directed segments u->v separately for each mode label in `trip_mode`.

    Assumes stop_times.txt is sorted by (trip_id, stop_sequence); the assumption is
    re-checked over every kept row while we are already reading them.
    Returns (edge_counts, stop_calls, stats).
    """
    modes = sorted(set(trip_mode.values()))
    edge_counts = {m: defaultdict(int) for m in modes}
    stop_calls = {m: defaultdict(int) for m in modes}
    trips_seen = {m: set() for m in modes}
    rows_read = rows_kept = self_loops = seq_regressions = 0
    t0 = time.time()

    with open(path, encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f)
        header = next(reader)
        for field in ('trip_id', 'stop_id', 'stop_sequence'):
            if field not in header:
                raise ValueError(f'stop_times.txt has no {field} column: {header}')
        ti = header.index('trip_id')
        si = header.index('stop_id')
        qi = header.index('stop_sequence')

        prev_trip, prev_stop, prev_seq, mode = None, None, None, None
        for row in reader:
            rows_read += 1
            if progress_every and rows_read % progress_every == 0:
                print(f'    {rows_read:,} rows | bus segments so far '
                      f'{len(edge_counts["bus"]):,} | {time.time() - t0:,.0f}s')
            trip = row[ti]
            if trip != prev_trip:
                # new trip block: decide its mode once, reset the per-trip state
                prev_trip, prev_stop, prev_seq = trip, None, None
                mode = trip_mode.get(trip)
                if mode is not None:
                    trips_seen[mode].add(trip)
            if mode is None:
                continue                      # a trip of some other mode
            rows_kept += 1
            stop = row[si]
            stop_calls[mode][stop] += 1
            try:
                seq = int(row[qi])
            except (ValueError, IndexError):
                seq = None
            if prev_stop is not None:
                if prev_stop != stop:
                    edge_counts[mode][(prev_stop, stop)] += 1
                else:
                    self_loops += 1
                if seq is not None and prev_seq is not None and seq <= prev_seq:
                    seq_regressions += 1
            prev_stop, prev_seq = stop, seq

    stats = {
        'rows_scanned': rows_read,
        'rows_kept': rows_kept,
        'kept_share_of_feed': round(rows_kept / max(rows_read, 1), 4),
        'self_loops_skipped': self_loops,
        'stop_sequence_regressions': seq_regressions,
        'seconds': round(time.time() - t0, 1),
    }
    for m in modes:
        stats[f'{m}_trips_streamed'] = len(trips_seen[m])
        stats[f'{m}_directed_segments'] = len(edge_counts[m])
        stats[f'{m}_stops_touched'] = len(stop_calls[m])
    return edge_counts, stop_calls, stats


edge_counts, stop_calls, stream_stats = stream_mode_edges(STOP_TIMES, TRIP_MODE)
print(json.dumps(stream_stats, indent=2))
if stream_stats['stop_sequence_regressions']:
    print('\n' + '!' * 78)
    print('WARNING: stop_sequence regressions were found inside trip blocks.')
    print('The feed is not sorted by (trip_id, stop_sequence), so the segments built')
    print('below connect stops that are not actually consecutive. Sort the file first.')
    print('!' * 78)

## 8. Building the bus graph

The counted segments become `networkx` objects: `D_bus`, the directed trip-adjacency graph (edge weight = number of bus trips on that segment), and `G_bus`, its undirected projection where the two travel directions are summed, so an undirected weight keeps its physical meaning of "total bus services crossing this link in either direction". All the connectivity work below (components, cut vertices, node removal) is direction-independent and uses `G_bus`.

Node attributes - Hebrew name, coordinates, district, metropolitan area - are copied from stage 02's `nodes.csv`. `_num` and `_txt` safely convert empty values and `NaN`, so a missing coordinate becomes `None` rather than a silent `NaN` that would later be plotted at a meaningless location. Bus stations carrying no attributes are counted and reported; the number should be ~0 because every bus station with a segment is also a node in the all-modes graph.

In [ ]:
# --- Build the bus graphs and attach stop attributes ----------------------
def build_graphs(counts):
    """Directed trip-adjacency graph plus its summed undirected projection."""
    D = nx.DiGraph()
    for (u, v), w in counts.items():
        D.add_edge(u, v, weight=int(w))
    G = nx.Graph()
    for u, v, data in D.edges(data=True):
        if G.has_edge(u, v):
            G[u][v]['weight'] += data['weight']
        else:
            G.add_edge(u, v, weight=data['weight'])
    return G, D


def _num(value):
    """Coerce to float; None for blanks, NaN or non-numeric input."""
    try:
        f = float(value)
    except (TypeError, ValueError):
        return None
    return None if not np.isfinite(f) else f


def _txt(value):
    """Coerce to a plain string; NaN and None become an empty string."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return ''
    return str(value)


ATTR = {str(r.get('stop_id')): {'stop_name': _txt(r.get('stop_name')),
                                'lat': _num(r.get('lat')),
                                'lon': _num(r.get('lon')),
                                'region': _txt(r.get('region')),
                                'metro': _txt(r.get('metro'))}
        for r in nodes_all.to_dict('records')}
DEFAULT_ATTR = {'stop_name': '', 'lat': None, 'lon': None, 'region': '', 'metro': ''}

G_bus, D_bus = build_graphs(edge_counts['bus'])
for graph in (G_bus, D_bus):
    for n in graph.nodes():
        graph.nodes[n].update(ATTR.get(n, DEFAULT_ATTR))

missing_attr = [n for n in G_bus.nodes() if n not in ATTR]
stops_without_segments = sorted(set(stop_calls['bus']) - set(G_bus.nodes()))

print(f'bus undirected : {G_bus.number_of_nodes():,} nodes, {G_bus.number_of_edges():,} edges')
print(f'bus directed   : {D_bus.number_of_nodes():,} nodes, {D_bus.number_of_edges():,} edges')
print(f'bus stops served by a trip but with no segment (excluded from V): '
      f'{len(stops_without_segments):,}')
print(f'bus nodes with no attributes in stage-02 nodes.csv: {len(missing_attr):,}')

## 9. The global structure of the bus-only network

The same set of descriptive metrics notebook 03 applied to the all-modes network, now on the bus alone: size, density, degree summary, connected components and the share of stations in the largest one, and the two classic local-connectivity structures - articulation points (nodes whose removal disconnects the graph) and bridges (edges whose removal disconnects it). Both are linear-time DFS routines in `networkx` (biconnected components via Hopcroft-Tarjan, chain decomposition for bridges) and finish in seconds even at this size. `articulation_points` is wrapped in a `set` because older versions of `networkx` may emit the same cut vertex more than once, which would inflate the count.

These numbers are the left column of the bus-versus-all-modes comparison in section 11.

In [ ]:
# --- Global structure of the bus network ----------------------------------
def structure_summary(G, D=None):
    """Size / sparsity / fragmentation / single-point-of-failure counts."""
    degrees = np.array([d for _, d in G.degree()])
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    largest = len(components[0])
    ap = set(nx.articulation_points(G))
    br = list(nx.bridges(G))
    summary = {
        'nodes': G.number_of_nodes(),
        'edges_undirected': G.number_of_edges(),
        'edges_directed': D.number_of_edges() if D is not None else None,
        'density': round(nx.density(G), 6),
        'avg_degree': round(float(degrees.mean()), 3),
        'median_degree': float(np.median(degrees)),
        'max_degree': int(degrees.max()),
        'share_degree_le_2': round(float((degrees <= 2).mean()), 4),
        'connected_components': len(components),
        'largest_component_nodes': largest,
        'largest_component_share': round(largest / G.number_of_nodes(), 4),
        'articulation_points': len(ap),
        'articulation_point_share': round(len(ap) / G.number_of_nodes(), 4),
        'bridges': len(br),
        'bridge_share': round(len(br) / G.number_of_edges(), 4),
    }
    return summary, ap, br


bus_stats, ap_bus, bridges_bus = structure_summary(G_bus, D_bus)
deg_bus = dict(G_bus.degree())

for key, value in bus_stats.items():
    print(f'{key:28s} {value}')
pd.DataFrame([bus_stats])

## 10. Centrality on the bus network

Four metrics, each answering a different question about a bus station:

* Degree - how many distinct stations are one segment away. Pure local connectivity.
* Weighted degree - the same, weighted by bus trips, i.e. how much service volume touches the station.
* PageRank on the directed weighted graph - importance in the sense of "where the service flow accumulates", resistant to the many degree-2 stations strung along a route.
* Sampled betweenness - what fraction of shortest paths pass through the station. This is the metric that identifies transit bottlenecks rather than busy termini.

Be honest about the betweenness. Exact betweenness requires a shortest-path sweep from each of about 29k nodes. We use `K_BETWEENNESS = 300` random pivots (fixed by `SEED`), and this is a sampled estimate: the top of the ranking is fairly stable, but individual values - especially small ones deep in the tail - carry real sampling error, and two stations with nearly identical estimates shouldn't be treated as if their order were reliable. That's why the column is named `approx_betweenness`, following notebook 04's convention. The metric is also hop-based (`weight=None`): a shortest path is the minimum number of segments, not the fastest trip, because travel times enter the project only in notebook 18.

The result is written to `tables/bus_station_metrics.csv`, the file later notebooks lean on.

In [ ]:
# --- Centrality on the bus graph (the expensive part is betweenness) ------
t0 = time.time()
wdeg_bus = dict(G_bus.degree(weight='weight'))
pagerank_bus = nx.pagerank(D_bus, weight='weight')
k_bus = min(K_BETWEENNESS, G_bus.number_of_nodes())
btw_bus = nx.betweenness_centrality(G_bus, k=k_bus, seed=SEED,
                                    normalized=True, weight=None)
print(f'centrality computed in {time.time() - t0:,.0f}s '
      f'(betweenness pivots: {k_bus} of {G_bus.number_of_nodes():,} nodes)')

bus_metrics = pd.DataFrame([{
    'stop_id': n,
    'stop_name': G_bus.nodes[n]['stop_name'],
    'lat': G_bus.nodes[n]['lat'],
    'lon': G_bus.nodes[n]['lon'],
    'region': G_bus.nodes[n]['region'],
    'degree': int(deg_bus[n]),
    'weighted_degree': int(wdeg_bus[n]),
    'approx_betweenness': float(btw_bus[n]),
    'is_articulation_point': n in ap_bus,
    'metro': G_bus.nodes[n]['metro'],
    'pagerank': float(pagerank_bus[n]),
    'in_degree': int(D_bus.in_degree(n)),
    'out_degree': int(D_bus.out_degree(n)),
    'bus_stop_calls': int(stop_calls['bus'].get(n, 0)),
} for n in G_bus.nodes()])

for col in ('degree', 'weighted_degree', 'approx_betweenness', 'pagerank'):
    bus_metrics['rank_' + col] = (bus_metrics[col]
                                  .rank(ascending=False, method='min').astype(int))

bus_metrics = (bus_metrics.sort_values('approx_betweenness', ascending=False)
               .reset_index(drop=True))
bus_metrics.to_csv(TABLES / 'bus_station_metrics.csv', index=False, encoding='utf-8-sig')
print('saved:', TABLES / 'bus_station_metrics.csv')
bus_metrics.head(10)

## 11. The top bus stations

Different metrics name different stations, so instead of picking one ranking we take the union of the top `TOP_N` under all four metrics and show where each one sits in each metric. A station that's in the top-20 across all four is a true hub; a station that's in the top-20 only in betweenness is a bottleneck carrying little service itself - a very different kind of criticality, and the more fragile of the two.

The figure shows the two most easily interpretable rankings side by side: weighted degree (service volume) and sampled betweenness (path bottleneck). Hebrew names are drawn through the bidi patch installed in section 3.

In [ ]:
# --- Top bus stations under four rankings ---------------------------------
rank_cols = ['rank_degree', 'rank_weighted_degree', 'rank_approx_betweenness',
             'rank_pagerank']
top_mask = (bus_metrics[rank_cols] <= TOP_N).any(axis=1)
top_table = (bus_metrics[top_mask]
             .sort_values('rank_approx_betweenness')
             [['stop_id', 'stop_name', 'region', 'metro', 'degree', 'weighted_degree',
               'approx_betweenness', 'pagerank', 'is_articulation_point'] + rank_cols]
             .reset_index(drop=True))
top_table.to_csv(TABLES / 'top_bus_stations.csv', index=False, encoding='utf-8-sig')
print(f'{len(top_table)} distinct stations appear in at least one top-{TOP_N} list')
print(f"of which {int(top_table['is_articulation_point'].sum())} are also bus cut vertices")

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
panels = [('weighted_degree', 'Bus trips on incident segments', '#2563eb'),
          ('approx_betweenness', f'Sampled betweenness (k={k_bus})', '#7c3aed')]
for ax, (col, xlabel, colour) in zip(axes, panels):
    panel = bus_metrics.sort_values(col, ascending=False).head(TOP_N).iloc[::-1]
    labels = [f'{name} ({sid})' for name, sid in zip(panel['stop_name'], panel['stop_id'])]
    ax.barh(range(len(panel)), panel[col].to_numpy(), color=colour)
    ax.set_yticks(range(len(panel)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel(xlabel)
    ax.set_title(f'Top {TOP_N} bus stations by {col.replace("_", " ")}')
plt.tight_layout()
plt.savefig(FIGURES / 'top_bus_stations.png', dpi=FIG_DPI)
plt.show()

top_table.head(TOP_N)

## 12. Bus cut vertices and cut edges

These are the structural single points of failure of the bus network: an articulation point is a station whose closure splits its component in two, and a bridge is a segment that's the only link between two parts of the network. Both tables are exported in full, enriched with name / coordinates / district / degree, so the report and the later lens-comparison notebook can cross-reference them.

Two caveats stated up front, the same ones that apply in notebook 03. First, this is a purely topological notion of failure - it ignores how many riders use a segment. Second, many cut vertices are low-degree stations on a dead-end branch, where "disconnects the network" means "leaves three isolated stations at the end of a route"; sorting by degree, as we do here, is what separates those from the hubs whose loss would isolate a whole sub-network.

In [ ]:
# --- Export bus cut vertices and cut edges --------------------------------
ap_table = (bus_metrics[bus_metrics['is_articulation_point']]
            [['stop_id', 'stop_name', 'lat', 'lon', 'region', 'metro',
              'degree', 'weighted_degree', 'approx_betweenness']]
            .sort_values('degree', ascending=False).reset_index(drop=True))
ap_table.to_csv(TABLES / 'bus_articulation_points.csv', index=False, encoding='utf-8-sig')

name_of = {n: (G_bus.nodes[n].get('stop_name') or n) for n in G_bus.nodes()}
bridge_table = pd.DataFrame([{
    'from_stop': u,
    'to_stop': v,
    'from_name': name_of[u],
    'to_name': name_of[v],
    'trip_frequency': int(G_bus[u][v]['weight']),
} for u, v in bridges_bus]).sort_values('trip_frequency', ascending=False).reset_index(drop=True)
bridge_table.to_csv(TABLES / 'bus_bridges.csv', index=False, encoding='utf-8-sig')

print(f"bus articulation points : {len(ap_table):,} "
      f"({len(ap_table) / G_bus.number_of_nodes():.2%} of bus stops)")
print(f"bus bridges             : {len(bridge_table):,} "
      f"({len(bridge_table) / G_bus.number_of_edges():.2%} of undirected segments)")
print(f"median degree of a bus cut vertex: {ap_table['degree'].median():.0f}")
print('\nHighest-degree bus cut vertices:')
display(ap_table.head(TOP_N))
print('Busiest bus bridges:')
bridge_table.head(TOP_N)

## 13. Resilience: targeted attack versus random failure

We delete stations from the bus graph and watch the largest connected component shrink. The reported size is the size of the largest surviving component as a share of the original node count, so the curve reflects both the stations we deleted and the ones they isolated.

Four removal orders are compared:

* `degree` - highest degree first (ties broken by weighted degree),
* `approx_betweenness` - highest sampled betweenness first,
* `articulation_first` - cut vertices first, sorted by degree, then everyone else by degree,
* `random` - uniform random order, averaged over `RANDOM_TRIALS` independent trials, recording the standard deviation across trials.

Two honest limitations. (1) The targeted rankings are computed once, on the full graph, and never recomputed as nodes disappear. A real adaptive attacker would recompute after each removal and do more damage; so our targeted curves are a lower bound on the achievable damage. (2) The betweenness order inherits `K_BETWEENNESS`'s sampling error, so it's a good order, not a proven-optimal one. Both choices are the cheap ones, and both bias the result against our own conclusion, which is the safe direction.

In [ ]:
# --- Targeted vs random node removal --------------------------------------
def resilience_curve(graph, order, counts):
    """Largest-component share (of the ORIGINAL node count) after removing order[:c]."""
    n0 = graph.number_of_nodes()
    all_nodes = set(graph.nodes())
    shares = []
    for c in counts:
        remaining = all_nodes - set(order[:c])
        if not remaining:
            shares.append(0.0)
            continue
        sizes = [len(comp) for comp
                 in nx.connected_components(graph.subgraph(remaining))]
        shares.append(max(sizes) / n0)
    return shares


strategies = {
    'degree': bus_metrics.sort_values(['degree', 'weighted_degree'],
                                      ascending=False)['stop_id'].tolist(),
    'approx_betweenness': bus_metrics.sort_values('approx_betweenness',
                                                  ascending=False)['stop_id'].tolist(),
    'articulation_first': bus_metrics.sort_values(['is_articulation_point', 'degree'],
                                                  ascending=False)['stop_id'].tolist(),
}

t0 = time.time()
rows = []
for name, order in strategies.items():
    for c, share in zip(REMOVAL_COUNTS, resilience_curve(G_bus, order, REMOVAL_COUNTS)):
        rows.append({'strategy': name, 'removed': c,
                     'largest_component_share': round(share, 5),
                     'std_across_trials': 0.0})

rng = random.Random(SEED)
node_list = list(G_bus.nodes())
random_curves = []
for _ in range(RANDOM_TRIALS):
    shuffled = node_list[:]
    rng.shuffle(shuffled)
    random_curves.append(resilience_curve(G_bus, shuffled, REMOVAL_COUNTS))
random_mean = np.mean(random_curves, axis=0)
random_std = np.std(random_curves, axis=0)
for c, m, s in zip(REMOVAL_COUNTS, random_mean, random_std):
    rows.append({'strategy': 'random', 'removed': c,
                 'largest_component_share': round(float(m), 5),
                 'std_across_trials': round(float(s), 5)})

resilience = pd.DataFrame(rows)
resilience.to_csv(TABLES / 'bus_resilience.csv', index=False, encoding='utf-8-sig')
print(f'removal experiment finished in {time.time() - t0:,.0f}s')

fig, ax = plt.subplots(figsize=(10, 6))
colours = {'degree': '#2563eb', 'approx_betweenness': '#7c3aed',
           'articulation_first': '#dc2626', 'random': '#64748b'}
for name, grp in resilience.groupby('strategy'):
    grp = grp.sort_values('removed')
    ax.plot(grp['removed'], grp['largest_component_share'], marker='o',
            color=colours.get(name), label=name)
    if name == 'random':
        ax.fill_between(grp['removed'],
                        grp['largest_component_share'] - grp['std_across_trials'],
                        grp['largest_component_share'] + grp['std_across_trials'],
                        color=colours['random'], alpha=0.25)
ax.set_xlabel('Bus stations removed')
ax.set_ylabel('Largest component / original number of stations')
ax.set_title('Bus network under targeted attack and random failure\n'
             f'(static rankings, {RANDOM_TRIALS} random trials, shaded band = 1 sd)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'bus_resilience_curves.png', dpi=FIG_DPI)
plt.show()

resilience.pivot(index='removed', columns='strategy',
                 values='largest_component_share')

## 14. Rebuilding the all-modes network as a comparison baseline

To answer "is bus interchangeable with the whole network?" you have to measure both networks the same way. So instead of quoting notebook 03's numbers, we rebuild the all-modes graph from stage 02's `edges.csv` (52k edges - a few seconds) and re-run exactly the same `structure_summary` function, plus a betweenness estimate with the same `k` and the same seed. Comparing our sampled bus betweenness against a column that was sampled differently in another notebook would mix a real difference with sampling noise.

Running betweenness here is the second expensive step (1-3 minutes); set `COMPUTE_ALLMODE_BETWEENNESS = False` in section 2 to skip it.

Where notebooks 03 and 04 are available we use them as an independent cross-check on our recomputation (matching the cut-vertex count, and the ranking correlation between our all-modes betweenness sample and theirs). If they don't exist, the notebook prints a message and continues - they aren't required.

In [ ]:
# --- Rebuild and re-measure the all-mode network --------------------------
D_all = nx.DiGraph()
for rec in edges_all.to_dict('records'):
    D_all.add_edge(str(rec['from_stop']), str(rec['to_stop']),
                   weight=int(rec.get('trip_frequency', 1)))
G_all = nx.Graph()
for u, v, data in D_all.edges(data=True):
    if G_all.has_edge(u, v):
        G_all[u][v]['weight'] += data['weight']
    else:
        G_all.add_edge(u, v, weight=data['weight'])
for n in G_all.nodes():
    G_all.nodes[n].update(ATTR.get(n, DEFAULT_ATTR))

all_stats, ap_all, bridges_all = structure_summary(G_all, D_all)
deg_all = dict(G_all.degree())

btw_all = None
if COMPUTE_ALLMODE_BETWEENNESS:
    t0 = time.time()
    btw_all = nx.betweenness_centrality(G_all, k=min(K_BETWEENNESS, G_all.number_of_nodes()),
                                        seed=SEED, normalized=True, weight=None)
    print(f'all-mode betweenness computed in {time.time() - t0:,.0f}s')
else:
    print('all-mode betweenness skipped (COMPUTE_ALLMODE_BETWEENNESS = False)')

# --- optional cross-checks against notebooks 03 and 04 --------------------
ap03 = try_load_stage_table('03', 'articulation_points.csv', '03_descriptive_analysis',
                            dtype={'stop_id': str})
if ap03 is not None:
    ap03_ids = set(ap03['stop_id'])
    print(f'cross-check vs notebook 03: it lists {len(ap03_ids):,} cut vertices, '
          f'we recomputed {len(ap_all):,}; {len(ap03_ids & ap_all):,} agree')

metrics04 = try_load_stage_table('04', 'stop_metrics.csv', '04_centrality_analysis',
                                 dtype={'stop_id': str})
if metrics04 is not None and btw_all is not None and 'approx_betweenness' in metrics04:
    joint = metrics04[['stop_id', 'approx_betweenness']].dropna()
    joint = joint[joint['stop_id'].isin(btw_all)]
    rho04, _ = spearmanr(joint['approx_betweenness'],
                         [btw_all[s] for s in joint['stop_id']])
    print(f'cross-check vs notebook 04: Spearman rho between its all-mode '
          f'approx_betweenness and ours = {rho04:.3f} over {len(joint):,} stops '
          '(two independent samples of the same quantity, so <1 is expected)')

comparison = pd.DataFrame({'metric': list(bus_stats.keys()),
                           'bus_only': list(bus_stats.values()),
                           'all_modes': [all_stats[k] for k in bus_stats]})
comparison['bus_share_of_all'] = [
    round(b / a, 4) if isinstance(b, (int, float)) and isinstance(a, (int, float)) and a
    else None
    for b, a in zip(comparison['bus_only'], comparison['all_modes'])]
comparison.to_csv(TABLES / 'bus_vs_allmode_summary.csv', index=False, encoding='utf-8-sig')
print('\nsaved:', TABLES / 'bus_vs_allmode_summary.csv')
comparison

## 15. Does removing the trains change which stations are cut vertices?

This is the question the notebook exists for, and the answer isn't obvious even though bus is 97% of trips: rail links are few but long, connecting cities that the bus network might connect only through a single corridor. Deleting a redundant path can create new cut vertices, so the bus-only network can easily have cut vertices that don't exist in the all-modes network.

Every station that's a cut vertex in at least one of the two networks is classified into one of four categories:

* `both` - a cut vertex in both networks; the non-bus layer is irrelevant for it,
* `bus_only` - not a cut vertex at the national level but yes in the bus-only network: here rail / light rail supplied the redundancy, and the all-modes view hides a bus vulnerability,
* `all_modes_only` - a cut vertex at the national level but not in the bus-only network: it disconnected something reached by a non-bus mode,
* `absent_from_bus_network` - an all-modes cut vertex that isn't a bus station at all (rail-only or light-rail-only).

We also report the Jaccard overlap between the two cut-vertex sets, the Spearman correlation between bus degree and all-modes degree across the shared stations, and the top-50 overlap between the two betweenness rankings. Those three numbers together say how closely the bus network tracks the national one - measured, not assumed.

In [ ]:
# --- Cut-vertex and ranking comparison ------------------------------------
bus_nodes = set(G_bus.nodes())
all_nodes = set(G_all.nodes())
shared_nodes = bus_nodes & all_nodes
bus_only_nodes = bus_nodes - all_nodes
non_bus_nodes = all_nodes - bus_nodes

ap_both = ap_bus & ap_all
ap_bus_only = ap_bus - ap_all
ap_all_only = (ap_all & bus_nodes) - ap_bus
ap_absent = ap_all - bus_nodes
jaccard_ap = len(ap_both) / max(len(ap_bus | ap_all), 1)


def _status(node):
    if node in ap_both:
        return 'both'
    if node in ap_bus_only:
        return 'bus_only'
    if node in ap_absent:
        return 'absent_from_bus_network'
    return 'all_modes_only'


ap_compare = pd.DataFrame([{
    'stop_id': n,
    'stop_name': (G_bus.nodes[n]['stop_name'] if n in bus_nodes
                  else G_all.nodes[n]['stop_name']),
    'region': (G_bus.nodes[n]['region'] if n in bus_nodes else G_all.nodes[n]['region']),
    'status': _status(n),
    'degree_bus': int(deg_bus.get(n, 0)),
    'degree_all_modes': int(deg_all.get(n, 0)),
    'is_articulation_point_bus': n in ap_bus,
    'is_articulation_point_all_modes': n in ap_all,
} for n in sorted(ap_bus | ap_all)])
ap_compare = ap_compare.sort_values(['status', 'degree_all_modes', 'degree_bus'],
                                    ascending=[True, False, False]).reset_index(drop=True)
ap_compare.to_csv(TABLES / 'articulation_point_comparison.csv', index=False,
                  encoding='utf-8-sig')

# degree agreement over the stops both networks contain
shared_sorted = sorted(shared_nodes)
rho_degree, _ = spearmanr([deg_bus[n] for n in shared_sorted],
                          [deg_all[n] for n in shared_sorted])
identical_degree = sum(1 for n in shared_sorted if deg_bus[n] == deg_all[n])

# undirected edge overlap
bus_edge_set = {tuple(sorted(e)) for e in G_bus.edges()}
all_edge_set = {tuple(sorted(e)) for e in G_all.edges()}

overlap = {
    'bus_nodes': len(bus_nodes),
    'all_mode_nodes': len(all_nodes),
    'shared_nodes': len(shared_nodes),
    'nodes_in_bus_not_in_all_modes': len(bus_only_nodes),
    'nodes_in_all_modes_not_in_bus': len(non_bus_nodes),
    'node_coverage_of_all_modes': round(len(shared_nodes) / len(all_nodes), 4),
    'undirected_edge_coverage_of_all_modes': round(
        len(bus_edge_set & all_edge_set) / len(all_edge_set), 4),
    'edges_in_bus_not_in_all_modes': len(bus_edge_set - all_edge_set),
    'spearman_degree_bus_vs_all_modes': round(float(rho_degree), 4),
    'stops_with_identical_degree': identical_degree,
    'share_with_identical_degree': round(identical_degree / len(shared_sorted), 4),
    'articulation_points_bus': len(ap_bus),
    'articulation_points_all_modes': len(ap_all),
    'articulation_points_both': len(ap_both),
    'articulation_points_bus_only': len(ap_bus_only),
    'articulation_points_all_modes_only': len(ap_all_only),
    'articulation_points_absent_from_bus': len(ap_absent),
    'articulation_point_jaccard': round(jaccard_ap, 4),
}

if btw_all is not None:
    top50_bus = set(bus_metrics.nsmallest(50, 'rank_approx_betweenness')['stop_id'])
    top50_all = {s for s, _ in sorted(btw_all.items(), key=lambda kv: kv[1],
                                      reverse=True)[:50]}
    rho_btw, _ = spearmanr([btw_bus[n] for n in shared_sorted],
                           [btw_all[n] for n in shared_sorted])
    overlap['spearman_betweenness_bus_vs_all_modes'] = round(float(rho_btw), 4)
    overlap['top50_betweenness_overlap'] = len(top50_bus & top50_all)

for key, value in overlap.items():
    print(f'{key:42s} {value}')
print('\nsaved:', TABLES / 'articulation_point_comparison.csv')
print('\nCut vertices that exist ONLY once the non-bus layer is removed '
      '(rail was the redundancy):')
display(ap_compare[ap_compare['status'] == 'bus_only'].head(TOP_N))
print('Cut vertices that stop being cut vertices on bus alone:')
ap_compare[ap_compare['status'] == 'all_modes_only'].head(TOP_N)

## 16. Three pictures of the same comparison

* Left - bus degree versus all-modes degree for every shared station, with the `y = x` line. Points on the line are stations the non-bus layer doesn't touch at all; points above it are stations that gain neighbors from rail / light rail. Because most stations have small degrees, the axes are logarithmic and the points are drawn with high transparency.
* Middle - the four cut-vertex categories as counts. The `bus_only` bar is the important one: these are vulnerabilities the all-modes analysis in notebook 03 can't see.
* Right - the two degree distributions on log-log axes. If the bus network really is nearly a copy of the national one, the two curves should be almost indistinguishable; any separation at the high-degree end is the multi-modal transfer termini.

In [ ]:
# --- Comparison figure ----------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

x = np.array([deg_bus[n] for n in shared_sorted], dtype=float)
y = np.array([deg_all[n] for n in shared_sorted], dtype=float)
axes[0].scatter(x, y, s=6, alpha=0.15, color='#2563eb', linewidths=0)
lim = max(x.max(), y.max())
axes[0].plot([1, lim], [1, lim], color='#111827', linestyle='--', linewidth=1,
             label='y = x (bus is the whole story)')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('Degree in the bus-only network')
axes[0].set_ylabel('Degree in the all-mode network')
axes[0].set_title(f'Degree agreement over {len(shared_sorted):,} shared stops\n'
                  f'Spearman rho = {rho_degree:.3f}')
axes[0].legend(fontsize=9)

buckets = ['both', 'bus_only', 'all_modes_only', 'absent_from_bus_network']
counts = [len(ap_both), len(ap_bus_only), len(ap_all_only), len(ap_absent)]
bars = axes[1].bar(range(len(buckets)), counts,
                   color=['#94a3b8', '#dc2626', '#f59e0b', '#64748b'])
axes[1].set_xticks(range(len(buckets)))
axes[1].set_xticklabels(buckets, rotation=20, ha='right', fontsize=9)
axes[1].set_ylabel('Number of stations')
axes[1].set_title('Where the cut vertices agree and disagree')
for bar, val in zip(bars, counts):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{val:,}',
                 ha='center', va='bottom', fontsize=10)
axes[1].margins(y=0.15)

for label, degmap, colour in [('bus only', deg_bus, '#2563eb'),
                              ('all modes', deg_all, '#dc2626')]:
    counts_by_deg = pd.Series(list(degmap.values())).value_counts().sort_index()
    counts_by_deg = counts_by_deg[counts_by_deg.index > 0]
    axes[2].scatter(counts_by_deg.index, counts_by_deg.to_numpy(), s=14, alpha=0.7,
                    color=colour, label=label)
axes[2].set_xscale('log')
axes[2].set_yscale('log')
axes[2].set_xlabel('Degree')
axes[2].set_ylabel('Number of stations')
axes[2].set_title('Degree distribution, bus vs all modes')
axes[2].legend()

plt.tight_layout()
plt.savefig(FIGURES / 'bus_vs_allmode.png', dpi=FIG_DPI)
plt.show()

## 17. Demand-responsive bus (`route_type = 715`), reported separately

The feed contains a second bus-like mode: 14 routes / 458 trips of demand-responsive service. It is not merged into the bus network above, for two reasons. Structurally, its "segments" describe a flexible service pattern rather than a fixed line, so an edge there doesn't mean the same thing as an edge on a scheduled line. Practically, at 458 trips out of 412k it couldn't change any aggregate, but it could quietly add stations and edges that the report would then have to explain.

So we measure it on its own - how many stations it touches, how many segments it has, how fragmented it is, and how many of its stations already get regular bus service - and write the result to a one-row table of its own. If the overlap with the regular bus network is high, this layer adds essentially nothing to connectivity; if it's low, it serves places scheduled bus doesn't reach, and that's a finding in itself for the notebooks concerned with equity.

In [ ]:
# --- The demand-responsive layer, kept separate ---------------------------
drt_counts = edge_counts.get('drt', {})
if drt_counts:
    G_drt, D_drt = build_graphs(drt_counts)
    drt_components = sorted(nx.connected_components(G_drt), key=len, reverse=True)
    drt_nodes = set(G_drt.nodes())
    drt_summary = {
        'route_type': DRT_ROUTE_TYPE,
        'mode_label': MODE_LABELS[DRT_ROUTE_TYPE],
        'trips_streamed': stream_stats.get('drt_trips_streamed', 0),
        'stops': G_drt.number_of_nodes(),
        'directed_edges': D_drt.number_of_edges(),
        'undirected_edges': G_drt.number_of_edges(),
        'connected_components': len(drt_components),
        'largest_component_nodes': len(drt_components[0]),
        'stops_also_served_by_bus': len(drt_nodes & bus_nodes),
        'stops_unique_to_this_mode': len(drt_nodes - bus_nodes),
        'share_of_stops_shared_with_bus': round(len(drt_nodes & bus_nodes)
                                                / len(drt_nodes), 4),
        'edges_shared_with_bus': len({tuple(sorted(e)) for e in G_drt.edges()}
                                     & bus_edge_set),
    }
else:
    G_drt = nx.Graph()
    drt_summary = {'route_type': DRT_ROUTE_TYPE,
                   'mode_label': MODE_LABELS[DRT_ROUTE_TYPE],
                   'trips_streamed': 0, 'stops': 0, 'directed_edges': 0,
                   'undirected_edges': 0, 'connected_components': 0,
                   'largest_component_nodes': 0, 'stops_also_served_by_bus': 0,
                   'stops_unique_to_this_mode': 0,
                   'share_of_stops_shared_with_bus': None,
                   'edges_shared_with_bus': 0}
    print('No demand-responsive segments were found in the feed.')

pd.DataFrame([drt_summary]).to_csv(TABLES / 'demand_responsive_summary.csv',
                                   index=False, encoding='utf-8-sig')
for key, value in drt_summary.items():
    print(f'{key:34s} {value}')
print('\nsaved:', TABLES / 'demand_responsive_summary.csv')

## 18. Saving the stage outputs

Three things are written here, so no downstream step has to stream the 816 MB feed again:

* `bus_graph_undirected.pkl` - the bus graph itself, with node attributes attached. Pickles are fragile across versions, so the same information is also exported as plain CSV.
* `tables/bus_edges.csv` - the directed bus segments (`from_stop, to_stop, trip_frequency`), the portable form of the graph.
* `bus_summary.json` - every key number produced above plus the constants that produced it, so any result can always be traced back to the settings that created it.

Files are written only under `outputs/nb/15_bus_network/`.

In [ ]:
# --- Persist the graph, the edge list and the summary ---------------------
with open(STAGE / 'bus_graph_undirected.pkl', 'wb') as f:
    pickle.dump(G_bus, f)

bus_edges_df = pd.DataFrame([{'from_stop': u, 'to_stop': v,
                              'trip_frequency': int(data['weight'])}
                             for u, v, data in D_bus.edges(data=True)])
bus_edges_df.to_csv(TABLES / 'bus_edges.csv', index=False, encoding='utf-8-sig')

resilience_headline = {
    row['strategy'] + f"_share_at_{row['removed']}_removed":
        row['largest_component_share']
    for _, row in resilience.iterrows()
    if row['removed'] in (100, 1000, 3000)
}

bus_summary = {
    'constants': {
        'BUS_ROUTE_TYPE': BUS_ROUTE_TYPE,
        'DRT_ROUTE_TYPE': DRT_ROUTE_TYPE,
        'K_BETWEENNESS': K_BETWEENNESS,
        'COMPUTE_ALLMODE_BETWEENNESS': COMPUTE_ALLMODE_BETWEENNESS,
        'SEED': SEED,
        'RANDOM_TRIALS': RANDOM_TRIALS,
        'REMOVAL_COUNTS': REMOVAL_COUNTS,
        'TOP_N': TOP_N,
    },
    'mode_inventory': inventory.to_dict('records'),
    'stream_stats': stream_stats,
    'bus_structure': bus_stats,
    'all_mode_structure': all_stats,
    'bus_vs_all_mode': overlap,
    'resilience_headline': resilience_headline,
    'demand_responsive': drt_summary,
}
with open(STAGE / 'bus_summary.json', 'w', encoding='utf-8') as f:
    json.dump(bus_summary, f, ensure_ascii=False, indent=2, default=str)

written = [STAGE / 'bus_graph_undirected.pkl', STAGE / 'bus_summary.json',
           TABLES / 'bus_station_metrics.csv', TABLES / 'bus_edges.csv',
           TABLES / 'top_bus_stations.csv', TABLES / 'bus_articulation_points.csv',
           TABLES / 'bus_bridges.csv', TABLES / 'bus_resilience.csv',
           TABLES / 'bus_vs_allmode_summary.csv',
           TABLES / 'articulation_point_comparison.csv',
           TABLES / 'demand_responsive_summary.csv',
           FIGURES / 'top_bus_stations.png', FIGURES / 'bus_resilience_curves.png',
           FIGURES / 'bus_vs_allmode.png']
print('Written:')
for p in written:
    flag = 'ok ' if p.exists() else 'MISSING '
    size = f'{p.stat().st_size / 1024:,.0f} KB' if p.exists() else ''
    print(f'  {flag}{p}  {size}')

## Conclusions

Read these against the numbers printed above; the notebook is written so that every claim here can be checked from a table it just wrote.

* Bus really is nearly the whole network - and the notebook checks this instead of assuming it. The mode inventory in section 6 confirms that `route_type = 3` carries about 97% of all scheduled trips, and section 15 quantifies what that means structurally: the node-coverage and undirected-edge-coverage figures, the Spearman correlation between bus degree and all-modes degree, and the share of stations whose degree is identical in both graphs. Where the identical-degree share is high, the matching stations are ones the non-bus layer never touches, and any all-modes result about them carries over to the bus unchanged.
* But the word "nearly" carries weight, and it shows up precisely at the cut vertices. The `bus_only` category in `articulation_point_comparison.csv` counts stations that are not single points of failure in the national graph but are in the bus-only one - places where a rail or light-rail link quietly supplied the only alternative route. These are invisible to notebook 03 and they're the concrete answer to "does removing the trains change which stations are cut vertices?". Symmetrically, the `all_modes_only` and `absent_from_bus_network` categories are cut vertices that exist only because non-bus stations are in the graph. If the two disagreement categories come out small relative to `both`, the honest conclusion is that the two networks agree on most single points of failure and still disagree on a specific, nameable list - and that list, not the aggregate, is the useful output.
* Targeted removal beats random removal by a wide margin, as expected from a heavy-tailed spatial network - but note the two caveats framed in section 13: the attack rankings are static (never recomputed after a removal), so the targeted curves understate what an adaptive attacker could do, and the betweenness order rests on a 300-pivot sample. The random baseline is the more reliable of the two curves, because it needs no ranking at all.
* Sampled betweenness is an estimate, and its tail is noise. With `K_BETWEENNESS = 300` pivots out of about 29k possible sources, the top stations are reliable and the small values aren't. Any downstream notebook consuming `approx_betweenness` from `bus_station_metrics.csv` should use it to rank the head of the distribution, not for fine comparisons between mid-table stations. The optional cross-check against notebook 04 in section 14 is exactly that: two independent samples of the same size, so a Spearman rho below 1 is expected and is a measure of the sampling noise, not of disagreement.
* Bus criticality is topological, not demand-based. Everything done here counts trips and segments; nobody boards a bus in this model. A degree-2 cut vertex at the end of a rural branch and a degree-30 transfer terminus both count as "one cut vertex" until a demand weight is attached, which is what the socio-economic and demand-weighting notebooks add. The degree column in `bus_articulation_points.csv` is the first, coarse filter.
* Demand-responsive service (`route_type = 715`) is a rounding error in connectivity terms - 14 routes and 458 trips - and it's reported in its own separate table rather than folded into the bus graph. Its one interesting number is the share of its stations that regular bus already serves: a low share would mean it reaches places scheduled bus doesn't, which matters for equity even if it can't matter topologically at that volume.
* What this notebook can't tell you. It's a single static snapshot of the whole feed period, with no time-of-day breakdown (notebooks 19-20), no travel times (notebook 18), and no notion of a rider who can walk 100 m to another station - two stations on opposite sides of the same junction are separate, unconnected nodes unless a bus actually travels between them. That last assumption makes the network look more fragile than it is on the ground, and it applies to the cut-vertex counts here exactly as it does in notebook 03.